# LeetCode 876 — Middle of the Linked List bằng Python

**Link đề:** https://leetcode.com/problems/middle-of-the-linked-list/
**Độ khó:** Easy · **Chủ đề:** Linked List, Two Pointers

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def middleNode(self, head: ListNode | None) -> ListNode | None:
        ...
```

## Mục tiêu học
1. Hiểu linked list đơn: node, `val`, `next`, `head`, `None`; vì sao **không** truy cập được `head[k]` như list.
2. Xác định chính xác "node giữa" với cả độ dài lẻ và chẵn (đề chọn **giữa thứ hai**) bằng công thức chỉ số `n // 2`.
3. Đi từ cách trực giác (chép ra mảng, đếm độ dài hai lượt) đến kỹ thuật **hai con trỏ nhanh–chậm** (slow/fast pointers).
4. Trace từng bước, chứng minh invariant `fast` đi gấp đôi `slow`, phân tích Time/Space.
5. Biết biến thể: lấy giữa **thứ nhất**, xóa node giữa, phát hiện chu trình… và làm 6 bài tập.

## 1. Đề bài và ví dụ

Cho `head` của một linked list đơn, trả về **node ở giữa**. Nếu có **hai** node giữa, trả về node **thứ hai**.

| Input | Output (danh sách bắt đầu từ node trả về) | Giải thích |
|---|---|---|
| `[1,2,3,4,5]` | `[3,4,5]` | 5 node, giữa là node 3 |
| `[1,2,3,4,5,6]` | `[4,5,6]` | 6 node, hai node giữa là 3 và 4 → chọn 4 |

**Constraints:** số node trong `[1, 100]`; `1 <= Node.val <= 100`.

Chú ý: đề trả về **node** (không phải giá trị). Output `[3,4,5]` là cách LeetCode in danh sách **bắt đầu từ node đó** đến hết.

## 2. Nền tảng: `ListNode` và các hàm phụ

Một node có `val` và `next` (node kế hoặc `None`). Ta tự định nghĩa class và 3 hàm học tập:
- `build_list(values)`: dựng linked list từ list Python.
- `to_list(head)`: đi theo `next`, gom giá trị.
- `show(head)`: in dạng mũi tên.

In [1]:
from typing import Optional

class ListNode:
    def __init__(self, val: int = 0, next: "Optional[ListNode]" = None):
        self.val = val
        self.next = next

def build_list(values: list[int]) -> Optional[ListNode]:
    head = None
    for v in reversed(values):          # dựng từ cuối: node mới trỏ về head cũ
        head = ListNode(v, head)
    return head

def to_list(head: Optional[ListNode]) -> list[int]:
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out

def show(head: Optional[ListNode]) -> str:
    return " -> ".join(map(str, to_list(head))) + " -> None" if head else "None"

h = build_list([1, 2, 3, 4, 5])
print(show(h))
print("head.val =", h.val, "| head.next.val =", h.next.val)

1 -> 2 -> 3 -> 4 -> 5 -> None
head.val = 1 | head.next.val = 2


### 2.1. Vì sao không viết được `head[2]`?

List Python lưu phần tử **liền nhau** trong bộ nhớ nên truy cập `a[k]` là `O(1)`. Linked list thì mỗi node chỉ biết **node kế tiếp**. Muốn tới node thứ `k` phải đi `k` bước `cur = cur.next` → `O(k)`.

Và ta cũng **không biết trước độ dài** — phải đi đến `None` mới biết.

In [2]:
def get_kth(head: Optional[ListNode], k: int) -> ListNode:
    cur = head
    for _ in range(k):       # đi k bước
        cur = cur.next
    return cur

print("Node chỉ số 2:", get_kth(h, 2).val)
try:
    h[2]
except TypeError as e:
    print("Lỗi khi viết head[2]:", e)

Node chỉ số 2: 3
Lỗi khi viết head[2]: 'ListNode' object is not subscriptable


### 2.2. "Node giữa" tính theo chỉ số: luôn là `n // 2`

Đánh chỉ số từ 0. Với `n` node:

| n | Các chỉ số | Node giữa theo đề | `n // 2` |
|---:|---|---|---:|
| 1 | 0 | 0 | 0 |
| 2 | 0 1 | 1 (giữa thứ hai) | 1 |
| 5 | 0 1 **2** 3 4 | 2 | 2 |
| 6 | 0 1 2 **3** 4 5 | 3 (giữa thứ hai) | 3 |

`//` là **chia lấy phần nguyên** (floor division): `5 // 2 = 2`, `6 // 2 = 3`. Công thức `n // 2` tự động chọn "giữa thứ hai" khi `n` chẵn. Nếu đề muốn "giữa thứ nhất" thì là `(n - 1) // 2`.

In [3]:
for n in range(1, 8):
    print(f"n={n}: giữa thứ hai = n//2 = {n // 2} | giữa thứ nhất = (n-1)//2 = {(n - 1) // 2}")

n=1: giữa thứ hai = n//2 = 0 | giữa thứ nhất = (n-1)//2 = 0
n=2: giữa thứ hai = n//2 = 1 | giữa thứ nhất = (n-1)//2 = 0
n=3: giữa thứ hai = n//2 = 1 | giữa thứ nhất = (n-1)//2 = 1
n=4: giữa thứ hai = n//2 = 2 | giữa thứ nhất = (n-1)//2 = 1
n=5: giữa thứ hai = n//2 = 2 | giữa thứ nhất = (n-1)//2 = 2
n=6: giữa thứ hai = n//2 = 3 | giữa thứ nhất = (n-1)//2 = 2
n=7: giữa thứ hai = n//2 = 3 | giữa thứ nhất = (n-1)//2 = 3


## 3. Cách 1 — Chép các node vào list Python

Ý tưởng tự nhiên nhất: "linked list khó truy cập theo chỉ số → chuyển sang list Python".
1. Đi từ `head`, `append` **chính các node** (không chỉ giá trị) vào `nodes`.
2. Trả về `nodes[len(nodes) // 2]`.

Lưu node (không phải `val`) vì đề yêu cầu trả **node**.

In [4]:
def middle_array(head: Optional[ListNode]) -> Optional[ListNode]:
    nodes = []
    cur = head
    while cur is not None:
        nodes.append(cur)
        cur = cur.next
    return nodes[len(nodes) // 2]

print(show(middle_array(build_list([1, 2, 3, 4, 5]))))
print(show(middle_array(build_list([1, 2, 3, 4, 5, 6]))))

3 -> 4 -> 5 -> None
4 -> 5 -> 6 -> None


**Độ phức tạp:** thời gian `O(n)`, bộ nhớ `O(n)` cho list `nodes`. Đúng và dễ hiểu, nhưng tốn thêm bộ nhớ tỉ lệ với độ dài.

## 4. Cách 2 — Đếm độ dài rồi đi `n // 2` bước (hai lượt duyệt)

Bỏ list phụ đi:
1. **Lượt 1:** đếm `n`.
2. **Lượt 2:** từ `head` đi `n // 2` bước.

Bộ nhớ chỉ còn vài biến → `O(1)`. Nhưng phải duyệt danh sách **hai lần** (khoảng `n + n/2` bước).

In [5]:
def length(head: Optional[ListNode]) -> int:
    n, cur = 0, head
    while cur:
        n += 1
        cur = cur.next
    return n

def middle_two_pass(head: Optional[ListNode]) -> Optional[ListNode]:
    n = length(head)
    cur = head
    for _ in range(n // 2):
        cur = cur.next
    return cur

print(show(middle_two_pass(build_list([1, 2, 3, 4, 5]))))
print(show(middle_two_pass(build_list([1, 2, 3, 4, 5, 6]))))

3 -> 4 -> 5 -> None
4 -> 5 -> 6 -> None


## 5. Nút thắt và insight

Câu hỏi: *"Có cách nào chỉ duyệt **một lần**, không dùng mảng phụ, mà vẫn biết đâu là giữa?"*

Vấn đề là: khi đang đi, ta không biết `n`. Nhưng hãy tưởng tượng **hai người** chạy trên cùng một đường:
- Người chậm (`slow`) mỗi lần đi **1** bước.
- Người nhanh (`fast`) mỗi lần đi **2** bước.

Khi người nhanh **về đích**, người chậm mới đi được **một nửa quãng đường** → đang đứng **ở giữa**!

Đây là kỹ thuật **fast & slow pointers** (còn gọi "rùa và thỏ" — tortoise and hare).

### 5.1. Điều kiện dừng: `while fast and fast.next`

`fast` cần nhảy `fast = fast.next.next`. Để không lỗi, cần:
- `fast` khác `None` (để đọc `fast.next`),
- `fast.next` khác `None` (để đọc `fast.next.next`).

Python đánh giá `and` **từ trái sang phải và dừng sớm** (short-circuit): nếu `fast` là `None` thì không đọc `fast.next` nữa → không lỗi.

In [6]:
fast = None
print("fast and fast.next ->", fast and fast.next)   # dừng ở fast=None, không lỗi

try:
    print(fast.next and fast)                         # đọc .next của None -> lỗi
except AttributeError as e:
    print("Đảo thứ tự gây lỗi:", e)

fast and fast.next -> None
Đảo thứ tự gây lỗi: 'NoneType' object has no attribute 'next'


### 5.2. Tính tay với `n = 5` và `n = 6`

`n = 5` (`1→2→3→4→5`):

| Vòng | slow | fast | Điều kiện tiếp tục? |
|---:|---:|---:|---|
| bắt đầu | 1 | 1 | fast=1, fast.next=2 → tiếp |
| 1 | 2 | 3 | fast.next=4 → tiếp |
| 2 | 3 | 5 | fast.next=None → **dừng** |

→ `slow = 3` ✔.

`n = 6` (`1→…→6`):

| Vòng | slow | fast | Điều kiện tiếp tục? |
|---:|---:|---:|---|
| bắt đầu | 1 | 1 | tiếp |
| 1 | 2 | 3 | tiếp |
| 2 | 3 | 5 | fast.next=6 → tiếp |
| 3 | 4 | None | fast=None → **dừng** |

→ `slow = 4` ✔ (giữa thứ hai). Với `n` lẻ, `fast` dừng ở **node cuối**; với `n` chẵn, `fast` dừng ở **`None`**.

## 6. Cách 3 — Lời giải chính: fast & slow pointers

Ghép các mảnh:
1. `slow = fast = head`.
2. Trong khi `fast and fast.next`: `slow` đi 1 bước, `fast` đi 2 bước.
3. Trả về `slow`.

In [7]:
def middle_fast_slow(head: Optional[ListNode]) -> Optional[ListNode]:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next          # 1 bước
        fast = fast.next.next     # 2 bước
    return slow

for vals in ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6], [7], [7, 8]):
    print(vals, "->", show(middle_fast_slow(build_list(vals))))

[1, 2, 3, 4, 5] -> 3 -> 4 -> 5 -> None
[1, 2, 3, 4, 5, 6] -> 4 -> 5 -> 6 -> None
[7] -> 7 -> None
[7, 8] -> 8 -> None


### 6.1. `slow = fast = head` nghĩa là gì?

Gán **chuỗi**: cả hai tên cùng trỏ vào **một node** `head`. Không tạo bản sao node nào. Sau đó gán lại `slow = slow.next` chỉ đổi chỗ mà tên `slow` dán vào — **không** làm thay đổi danh sách.

In [8]:
head = build_list([1, 2, 3])
slow = fast = head
print("slow is fast is head:", slow is fast is head)
slow = slow.next
print("Sau slow = slow.next -> slow.val =", slow.val, "| head vẫn là:", show(head))

slow is fast is head: True
Sau slow = slow.next -> slow.val = 2 | head vẫn là: 1 -> 2 -> 3 -> None


## 7. TRACE từng bước

Hàm `trace_middle` in vị trí (chỉ số) và giá trị của `slow`, `fast` sau mỗi vòng. Thử cả độ dài lẻ và chẵn để thấy hai kiểu dừng khác nhau.

In [9]:
def trace_middle(values: list[int]) -> int:
    head = build_list(values)
    slow = fast = head
    si = fi = 0
    step = 0
    def pos(node, idx):
        return f"#{idx}({node.val})" if node else "None"
    print(f"List {values} (n={len(values)}), mong đợi chỉ số {len(values)//2}")
    print(f"  vòng {step}: slow={pos(slow, si)}, fast={pos(fast, fi)}")
    while fast and fast.next:
        slow, si = slow.next, si + 1
        fast, fi = fast.next.next, fi + 2
        step += 1
        print(f"  vòng {step}: slow={pos(slow, si)}, fast={pos(fast, fi)}")
    reason = "fast là None" if fast is None else "fast.next là None (fast ở node cuối)"
    print(f"  Dừng vì {reason} -> trả về node {slow.val} (chỉ số {si})")
    return slow.val

trace_middle([1, 2, 3, 4, 5]); print()
trace_middle([1, 2, 3, 4, 5, 6])

List [1, 2, 3, 4, 5] (n=5), mong đợi chỉ số 2
  vòng 0: slow=#0(1), fast=#0(1)
  vòng 1: slow=#1(2), fast=#2(3)
  vòng 2: slow=#2(3), fast=#4(5)
  Dừng vì fast.next là None (fast ở node cuối) -> trả về node 3 (chỉ số 2)

List [1, 2, 3, 4, 5, 6] (n=6), mong đợi chỉ số 3
  vòng 0: slow=#0(1), fast=#0(1)
  vòng 1: slow=#1(2), fast=#2(3)
  vòng 2: slow=#2(3), fast=#4(5)
  vòng 3: slow=#3(4), fast=None
  Dừng vì fast là None -> trả về node 4 (chỉ số 3)


4

### 7.1. Trường hợp biên: 1 node và 2 node

- `n = 1`: `fast.next` là `None` ngay từ đầu → không vào vòng → trả `head`.
- `n = 2`: vào 1 vòng, `slow` sang node 2, `fast` thành `None` → trả node 2 (giữa thứ hai) ✔.

In [10]:
trace_middle([9]); print()
trace_middle([9, 10])

List [9] (n=1), mong đợi chỉ số 0
  vòng 0: slow=#0(9), fast=#0(9)
  Dừng vì fast.next là None (fast ở node cuối) -> trả về node 9 (chỉ số 0)

List [9, 10] (n=2), mong đợi chỉ số 1
  vòng 0: slow=#0(9), fast=#0(9)
  vòng 1: slow=#1(10), fast=None
  Dừng vì fast là None -> trả về node 10 (chỉ số 1)


10

### 7.2. Đếm số bước: fast-slow có thực sự "một lượt"?

So sánh tổng số lần đi `.next` của cách 2 (hai lượt) và cách 3. Cách 3: `slow` đi `n//2` bước, `fast` đi khoảng `n` bước → tổng ~`1.5n` lần `.next`, tương đương cách 2! Ưu điểm thật sự của fast-slow là **chỉ một vòng lặp** và là **nền tảng** cho các bài khó hơn (phát hiện chu trình, chia đôi list trong merge sort, kiểm tra palindrome list), không phải là ít phép tính hơn hẳn.

In [11]:
def count_moves_two_pass(n):
    return n + n // 2                       # lượt đếm + lượt đi n//2

def count_moves_fast_slow(values):
    head = build_list(values)
    slow = fast = head
    moves = 0
    while fast and fast.next:
        slow = slow.next; moves += 1
        fast = fast.next.next; moves += 2
    return moves

for n in (5, 6, 100):
    print(f"n={n}: hai lượt = {count_moves_two_pass(n)} bước .next | fast-slow = {count_moves_fast_slow(list(range(n)))} bước .next")

n=5: hai lượt = 7 bước .next | fast-slow = 6 bước .next
n=6: hai lượt = 9 bước .next | fast-slow = 9 bước .next
n=100: hai lượt = 150 bước .next | fast-slow = 150 bước .next


## 8. Code nộp LeetCode

In [12]:
class Solution:
    def middleNode(self, head: Optional[ListNode]) -> Optional[ListNode]:
        slow = fast = head
        while fast and fast.next:
            slow = slow.next
            fast = fast.next.next
        return slow

sol = Solution()
print(show(sol.middleNode(build_list([1, 2, 3, 4, 5]))))
print(show(sol.middleNode(build_list([1, 2, 3, 4, 5, 6]))))

3 -> 4 -> 5 -> None
4 -> 5 -> 6 -> None


### 8.1. Các bản nộp khác (đều Accepted)

In [13]:
class SolutionArray:
    def middleNode(self, head: Optional[ListNode]) -> Optional[ListNode]:
        nodes = []
        while head:
            nodes.append(head)
            head = head.next
        return nodes[len(nodes) // 2]

class SolutionTwoPass:
    def middleNode(self, head: Optional[ListNode]) -> Optional[ListNode]:
        n, cur = 0, head
        while cur:
            n += 1
            cur = cur.next
        cur = head
        for _ in range(n // 2):
            cur = cur.next
        return cur

print(show(SolutionArray().middleNode(build_list([1, 2, 3, 4, 5, 6]))))
print(show(SolutionTwoPass().middleNode(build_list([1, 2, 3, 4, 5]))))

4 -> 5 -> 6 -> None
3 -> 4 -> 5 -> None


## 9. Vì sao fast & slow đúng?

**Invariant:** sau `k` vòng lặp, `slow` ở chỉ số `k`, `fast` ở chỉ số `2k` (nếu `2k ≤ n-1`; nếu `2k = n` thì `fast` là `None`).

Vòng lặp tiếp tục khi `fast` và `fast.next` tồn tại, tức `2k ≤ n - 2`, tức `2k + 2 ≤ n`.

Vòng lặp **dừng** ở `k` nhỏ nhất mà `2k + 2 > n`, tức `2k ≥ n - 1`:
- `n` lẻ (`n = 2m + 1`): dừng khi `2k ≥ 2m` → `k = m = n // 2` (fast ở node cuối, chỉ số `2m = n - 1`).
- `n` chẵn (`n = 2m`): dừng khi `2k ≥ 2m - 1` → `k = m = n // 2` (fast ở chỉ số `2m = n`, tức `None`).

Cả hai trường hợp: `slow` ở chỉ số `n // 2` — đúng là node giữa (giữa thứ hai khi chẵn). ∎

In [14]:
# Kiểm chứng invariant cho mọi n từ 1 đến 30
for n in range(1, 31):
    head = build_list(list(range(n)))     # giá trị = chỉ số
    assert middle_fast_slow(head).val == n // 2
print("slow luôn dừng ở chỉ số n // 2 với n = 1..30")

slow luôn dừng ở chỉ số n // 2 với n = 1..30


## 10. Độ phức tạp và so sánh các cách

`n` = số node.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Số lượt duyệt | Ghi chú |
|---|---|---|---|---:|---|
| 1. Mảng node | `nodes[n//2]` | `O(n)` | `O(n)` | 1 | Dễ nhất, tốn bộ nhớ |
| 2. Đếm rồi đi | `length` + đi `n//2` | `O(n)` | `O(1)` | 2 | Không mảng phụ |
| 3. Fast & slow | thỏ gấp đôi rùa | `O(n)` | `O(1)` | 1 | Chuẩn phỏng vấn, tái sử dụng nhiều |

Không cách nào nhanh hơn `O(n)` được: muốn biết đâu là giữa, phải "nhìn thấy" đến cuối danh sách.

## 11. Bộ test: mẫu đề, biên, đối chiếu các cách

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

methods = {
    "array": middle_array,
    "two_pass": middle_two_pass,
    "fast_slow": middle_fast_slow,
    "Solution": Solution().middleNode,
    "SolutionArray": SolutionArray().middleNode,
    "SolutionTwoPass": SolutionTwoPass().middleNode,
}

cases = [
    ("đề VD1 (lẻ)", [1, 2, 3, 4, 5], [3, 4, 5]),
    ("đề VD2 (chẵn)", [1, 2, 3, 4, 5, 6], [4, 5, 6]),
    ("1 node", [1], [1]),
    ("2 node", [1, 2], [2]),
    ("3 node", [1, 2, 3], [2, 3]),
    ("giá trị trùng", [5, 5, 5, 5], [5, 5]),
]
for name, vals, expected in cases:
    for mname, f in methods.items():
        assert to_list(f(build_list(vals))) == expected, (name, mname)
    check(name, to_list(middle_fast_slow(build_list(vals))), expected)

PASS  đề VD1 (lẻ): [3, 4, 5]
PASS  đề VD2 (chẵn): [4, 5, 6]
PASS  1 node: [1]
PASS  2 node: [2]
PASS  3 node: [2, 3]
PASS  giá trị trùng: [5, 5]


### 11.1. Phải trả về **đúng node**, không phải node mới cùng giá trị

Với `[5,5,5,5]`, so giá trị không đủ để biết đã trả đúng node. Dùng `is` để kiểm tra danh tính: node trả về phải **là** node chỉ số `n // 2` của danh sách gốc.

In [16]:
import random
random.seed(876)
for _ in range(300):
    n = random.randint(1, 100)
    vals = [random.randint(1, 3) for _ in range(n)]
    head = build_list(vals)
    target = get_kth(head, n // 2)
    for mname, f in methods.items():
        assert f(head) is target, mname
print("300 list ngẫu nhiên (n tới 100): mọi cách trả về ĐÚNG node (kiểm bằng `is`).")

300 list ngẫu nhiên (n tới 100): mọi cách trả về ĐÚNG node (kiểm bằng `is`).


## 12. Lỗi người mới hay gặp

### Lỗi 1: Điều kiện `while fast.next and fast` (đảo thứ tự)
Khi `n` chẵn, `fast` thành `None` → `fast.next` gây `AttributeError`. **Sửa:** kiểm `fast` trước.

In [17]:
def middle_wrong_order(head):
    slow = fast = head
    while fast.next and fast:            # SAI thứ tự
        slow = slow.next
        fast = fast.next.next
    return slow

try:
    middle_wrong_order(build_list([1, 2, 3, 4]))
except AttributeError as e:
    print("Lỗi với n chẵn:", e)
print("Với n lẻ thì 'may mắn' chạy được:", middle_wrong_order(build_list([1, 2, 3])).val)

Lỗi với n chẵn: 'NoneType' object has no attribute 'next'
Với n lẻ thì 'may mắn' chạy được: 2


### Lỗi 2: Khởi tạo `fast = head.next` (lệch một bước)
Nhiều tài liệu dùng `fast = head.next` — cách đó trả về **giữa thứ nhất** khi `n` chẵn, sai yêu cầu đề này.

In [18]:
def middle_first(head):
    slow, fast = head, head.next          # lệch: cho giữa THỨ NHẤT
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("[1..6] với fast=head.next ->", middle_first(build_list([1, 2, 3, 4, 5, 6])).val, "(đề cần 4)")
print("[1..5] với fast=head.next ->", middle_first(build_list([1, 2, 3, 4, 5])).val, "(vẫn đúng 3 khi lẻ -> lỗi khó phát hiện!)")

[1..6] với fast=head.next -> 3 (đề cần 4)
[1..5] với fast=head.next -> 3 (vẫn đúng 3 khi lẻ -> lỗi khó phát hiện!)


### Lỗi 3: Dùng `len(nodes) / 2` thay vì `//`
`/` trả về `float` (`3.0`) → `nodes[3.0]` lỗi `TypeError`; với `n` lẻ còn ra `2.5`.

### Lỗi 4: Trả `slow.val` thay vì `slow`
Đề yêu cầu trả **node**. Trả số nguyên sẽ sai kiểu khi LeetCode in danh sách từ đó.

### Lỗi 5: "Đi `n // 2 - 1` bước" do nhầm chỉ số bắt đầu từ 1
Nếu đếm node từ 1, giữa là node thứ `n // 2 + 1`, tức đi `n // 2` bước từ `head`. Luôn tự kiểm với `n = 1, 2`.

In [19]:
nodes = [10, 20, 30, 40]
try:
    nodes[len(nodes) / 2]
except TypeError as e:
    print("Dùng / :", e)
print("Dùng //:", nodes[len(nodes) // 2])

Dùng / : list indices must be integers or slices, not float
Dùng //: 30


## 13. Mẹo nhận dạng pattern

- Linked list + "giữa", "nửa", "chia đôi", "phần tử thứ k từ cuối", "có chu trình không" → nghĩ ngay **hai con trỏ** khác tốc độ hoặc cách nhau `k` bước.
- Đề cấm/ngầm khuyến khích bộ nhớ `O(1)` trên linked list → hai con trỏ thay cho mảng phụ.
- Nhớ cặp điều kiện: **`while fast and fast.next`** + khởi tạo `slow = fast = head` → giữa thứ hai.
- Giữa thứ nhất: `while fast.next and fast.next.next` (với `head` khác `None`) hoặc khởi tạo `fast = head.next`.

Bài cùng họ: 141/142 Linked List Cycle, 234 Palindrome Linked List, 143 Reorder List, 148 Sort List (chia đôi), 19 Remove Nth Node From End, 2095 Delete the Middle Node.

## 14. Biến thể

### 14.1. Giữa thứ nhất (dùng khi muốn **cắt đôi** list cho merge sort)
### 14.2. Xóa node giữa (LeetCode 2095)
Cần node **đứng trước** node giữa → cho `fast` đi trước thêm một bước (bắt đầu từ `head.next.next`).

In [20]:
def middle_first_variant(head):
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next
        fast = fast.next.next
    return slow

def delete_middle(head):
    if head.next is None:
        return None
    slow, fast = head, head.next.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    slow.next = slow.next.next            # slow là node đứng trước node giữa
    return head

print("Giữa thứ nhất [1..6]:", middle_first_variant(build_list([1, 2, 3, 4, 5, 6])).val)
print("Xóa giữa [1,3,4,7,1,2,6]:", show(delete_middle(build_list([1, 3, 4, 7, 1, 2, 6]))))
print("Xóa giữa [1,2,3,4]:", show(delete_middle(build_list([1, 2, 3, 4]))))

Giữa thứ nhất [1..6]: 3
Xóa giữa [1,3,4,7,1,2,6]: 1 -> 3 -> 4 -> 1 -> 2 -> 6 -> None
Xóa giữa [1,2,3,4]: 1 -> 2 -> 4 -> None


### 14.3. Phát hiện chu trình (Floyd)
Cùng cặp con trỏ: nếu có vòng, `fast` sẽ đuổi kịp `slow` (`fast is slow`); nếu không, `fast` chạm `None`.

In [21]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

a = build_list([1, 2, 3, 4])
print("Không vòng:", has_cycle(a))
get_kth(a, 3).next = get_kth(a, 1)        # node 4 -> node 2 tạo vòng
print("Có vòng   :", has_cycle(a))

Không vòng: False
Có vòng   : True


## 15. Bài tập tự luyện

Các hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
Với list `[10, 20, 30, 40, 50, 60, 70, 80]`, lập bảng `slow`/`fast` từng vòng. Node trả về là gì? Có bao nhiêu vòng lặp?

### Bài 2 — Dự đoán output
Nếu đổi điều kiện thành `while fast.next and fast.next.next` (khởi tạo vẫn `slow = fast = head`), với `[1,2,3,4,5,6]` thì trả về node nào? Với `[1,2,3,4,5]`?

### Bài 3 — Điền chỗ trống: fast & slow

In [22]:
def middle_exercise(head):
    slow = fast = head
    # TODO 1: điều kiện vòng lặp an toàn
    # TODO 2: slow đi 1 bước, fast đi 2 bước
    # TODO 3: trả về node đúng
    pass

# assert middle_exercise(build_list([1,2,3,4,5])).val == 3
# assert middle_exercise(build_list([1,2,3,4,5,6])).val == 4

### Bài 4 — Giá trị giữa của list Python
Viết `middle_value(nums)` cho list Python thường (không phải linked list), trả giá trị giữa thứ hai. Một dòng là đủ.

In [23]:
def middle_value(nums: list[int]) -> int:
    # TODO
    pass

# assert middle_value([1,2,3,4,5]) == 3
# assert middle_value([1,2,3,4,5,6]) == 4

### Bài 5 — Node thứ k từ cuối, một lượt duyệt
Cho `k` (1 ≤ k ≤ n), trả về node thứ `k` tính từ cuối (k=1 là node cuối). Gợi ý: hai con trỏ **cách nhau k bước**.

In [24]:
def kth_from_end(head, k: int):
    # TODO
    pass

# assert kth_from_end(build_list([1,2,3,4,5]), 2).val == 4
# assert kth_from_end(build_list([1,2,3,4,5]), 5).val == 1

### Bài 6 — Mini challenge: linked list palindrome
Trả `True` nếu dãy giá trị đọc xuôi/ngược như nhau. Dùng node giữa, đảo nửa sau rồi so sánh (bộ nhớ `O(1)`).

In [25]:
def is_palindrome_list(head) -> bool:
    # TODO: tìm giữa -> đảo nửa sau -> so sánh hai nửa
    pass

# assert is_palindrome_list(build_list([1,2,2,1])) is True
# assert is_palindrome_list(build_list([1,2,3,2,1])) is True
# assert is_palindrome_list(build_list([1,2])) is False

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | n = 8 chẵn | fast sẽ thành None | 4 vòng, slow dừng ở 50 |
| 2 | Điều kiện chặt hơn | dừng sớm hơn 1 vòng khi chẵn | chẵn → 3; lẻ → 3 |
| 3 | Thỏ và rùa | `while fast and fast.next` | `slow = slow.next; fast = fast.next.next` |
| 4 | Chỉ số `n // 2` | `len(nums)` | `return nums[len(nums) // 2]` |
| 5 | Cho `fast` đi trước k bước | rồi đi cùng nhau tới khi `fast` là None | `slow` lúc đó là đáp án |
| 6 | Ghép 3 kỹ năng | đảo list bằng `prev/cur` | so sánh nửa đầu với nửa sau đã đảo |

## 17. Đáp án tham khảo

**Bài 1:** n = 8. Vòng 1: slow=20, fast=30; vòng 2: slow=30, fast=50; vòng 3: slow=40, fast=70; vòng 4: slow=50, fast=None → dừng. Trả node `50` (chỉ số 4 = 8 // 2), 4 vòng.

**Bài 2:** Với `[1..6]`: vòng 1 slow=2, fast=3; vòng 2 slow=3, fast=5; fast.next.next = None → dừng → **3** (giữa thứ nhất). Với `[1..5]`: slow=3, fast=5 → fast.next None → dừng → **3**.

In [26]:
def ans_middle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

def ans_middle_value(nums):
    return nums[len(nums) // 2]

def ans_kth_from_end(head, k):
    fast = slow = head
    for _ in range(k):
        fast = fast.next
    while fast:
        slow = slow.next
        fast = fast.next
    return slow

def ans_reverse(head):
    prev, cur = None, head
    while cur:
        cur.next, prev, cur = prev, cur, cur.next
    return prev

def ans_is_palindrome_list(head) -> bool:
    mid = ans_middle(head)
    second = ans_reverse(mid)
    p1, p2 = head, second
    while p2:                       # nửa sau ngắn hơn hoặc bằng
        if p1.val != p2.val:
            return False
        p1, p2 = p1.next, p2.next
    return True

# Kiểm tra bài 1, 2 bằng code
assert ans_middle(build_list([10,20,30,40,50,60,70,80])).val == 50
assert middle_first_variant(build_list([1,2,3,4,5,6])).val == 3
assert middle_first_variant(build_list([1,2,3,4,5])).val == 3
assert ans_middle(build_list([1,2,3,4,5])).val == 3
assert ans_middle(build_list([1,2,3,4,5,6])).val == 4
assert ans_middle_value([1,2,3,4,5]) == 3
assert ans_middle_value([1,2,3,4,5,6]) == 4
assert ans_kth_from_end(build_list([1,2,3,4,5]), 2).val == 4
assert ans_kth_from_end(build_list([1,2,3,4,5]), 5).val == 1
assert ans_is_palindrome_list(build_list([1,2,2,1])) is True
assert ans_is_palindrome_list(build_list([1,2,3,2,1])) is True
assert ans_is_palindrome_list(build_list([1,2])) is False
assert ans_is_palindrome_list(build_list([7])) is True
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


Ghi chú bài 6: cách trên **làm thay đổi** danh sách gốc (nửa sau bị đảo). Trong phỏng vấn nên nói rõ điều này và (nếu cần) đảo lại nửa sau trước khi trả kết quả.

## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def middleNode(self, head):
        slow = fast = head                 # cùng xuất phát
        while fast and fast.next:          # kiểm fast TRƯỚC fast.next
            slow = slow.next               # rùa: 1 bước
            fast = fast.next.next          # thỏ: 2 bước
        return slow                        # giữa (thứ hai nếu chẵn)
```

**Câu thần chú:** *"Thỏ về đích thì rùa ở giữa."*

**Checklist trước khi nộp:**
- [ ] Khởi tạo `slow = fast = head` (không phải `fast = head.next`) để ra giữa **thứ hai**?
- [ ] Điều kiện `fast and fast.next` đúng thứ tự?
- [ ] Trả về **node** `slow`, không phải `slow.val`?
- [ ] Đã thử tay `n = 1`, `n = 2`, một `n` lẻ và một `n` chẵn?
- [ ] Nếu dùng mảng: chỉ số dùng `//`, không dùng `/`?